# Generative Reflection Combinatorial (GRC) kernel

In this notebook we provide examples of how tilings made with the GRC kernel, introduced in v1.4 can be used

In [ ]:
from hypertiling import HyperbolicTiling, HyperbolicGraph, TilingKernels, GraphKernels
from hypertiling.graphics.plot import plot_tiling

import matplotlib.cm as cmap
import numpy as np
import time

import hypertiling.ion as ion
ion.set_verbosity_level("Warning")

## Initialization

The GRC kernel can be accessed by the **HyperbolicTiling** and **HyperbolicGraph** factory pattern. In both cases, the same GRC class object is initalized. The difference between both calls are the default values given to the GRC class for initialization:

In general, GRC takes **three different keyword arguments** with the corresponding defaults

- sector = True: Controlls whether the whole tiling is constructed (False) or only one symmetry sector (True) with generators providing the remaining sectors
- tiling = False: Controlls whether cell coordinates are calculated (True)
- nbrs = False: Controlls whether the neighborhood relations are calculated (True)

On one hand, advantage of having coordinates is that we can plot the tilings. On the other hand, tilings require 
more time and memory to be generated. 

If GRC is called by the HyperbolicTiling factory pattern, the tiling keyword is adjusted such that tiling=True.
Similarily, if GRC is called by the HyperbolicGraph factory pattern, the graph keyword is adjusted such that graph=True.

Please note that in both cases, the other keyword can be manually set to either value. In case of tiling and nbrs being True, the results of HyperbolicTiling and HyperbolicGraph are identical.

In [ ]:

p, q, n = 5, 4, 12

def benchmark(func, **kwargs):
    """Measures execution time, averaging if the first run is below 10 ms."""
    t_start = time.time()
    func(p, q, n, **kwargs)
    dt = time.time() - t_start
    
    threshold = 0.01
    if dt < threshold:
        # Target a total sample window of ~0.1s for precision
        runs = 10
        t_start_loop = time.time()
        for _ in range(runs):
            func(p, q, n, **kwargs)
        return (time.time() - t_start_loop) / runs
    return dt

# --- Create some tilings ---
t0 = benchmark(HyperbolicTiling, kernel=TilingKernels.GRC, nbrs=False)
t1 = benchmark(HyperbolicTiling, kernel=TilingKernels.GRC, sector=False, nbrs=False)

print("Tilings:")
print(f"\tsector=True,  nbrs=False took {t0:.6f}s")
print(f"\tsector=False, nbrs=False took {t1:.6f}s")
print("=" * 39)

# --- Create some graphs ---
t2 = benchmark(HyperbolicGraph, kernel=GraphKernels.GRC, nbrs=False)
t3 = benchmark(HyperbolicGraph, kernel=GraphKernels.GRC, sector=False, nbrs=False)

print("Graphs:")
print(f"\tsector=True,  tiling=False took {t2:.6f}s")
print(f"\tsector=False, tiling=False took {t3:.6f}s")
print("=" * 39)

# --- Identical results check (tiling=True and nbrs=True) ---
# HyperbolicTiling pattern
t_tiling_true  = benchmark(HyperbolicTiling, kernel=TilingKernels.GRC, nbrs=True)
t_tiling_false = benchmark(HyperbolicTiling, kernel=TilingKernels.GRC, sector=False, nbrs=True)

# HyperbolicGraph pattern
t_graph_true  = benchmark(HyperbolicGraph, kernel=GraphKernels.GRC, tiling=True)
t_graph_false = benchmark(HyperbolicGraph, kernel=GraphKernels.GRC, sector=False, tiling=True)

print("Identical (tiling=True, nbrs=True):")
print("\tsector=True:")
print(f"\t\tHyperbolicTiling: {t_tiling_true:.6f}s")
print(f"\t\tHyperbolicGraph:  {t_graph_true:.6f}s")
print("\tsector=False:")
print(f"\t\tHyperbolicTiling: {t_tiling_false:.6f}s")
print(f"\t\tHyperbolicGraph:  {t_graph_false:.6f}s")

## Examples

### The hypertiling logo

In [ ]:
t = HyperbolicTiling(7, 3, 3, kernel=TilingKernels.GRC)

In [ ]:
ct = np.zeros(len(t))
for i, poly in enumerate(t):
    val = np.real(poly[0])-np.imag(poly[0])
    ct[i] = np.sign(val)*(np.abs(val))**1.4

In [ ]:
plot_tiling(t, ct, cmap=cmap.RdYlGn, edgecolor="w", lw=5, clim=[-1,1]);

### Tiling as Vector Graphics

In [ ]:
from hypertiling import HyperbolicTiling
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from hypertiling.graphics.svg import SVGCanvas, Tiling, UnitCircle, draw_svg

In [ ]:
# generate tiling
tiling = HyperbolicTiling(5, 4, 6, kernel="GRC")

# The svg API expects explicit hex strings for 'facecolors', so we process the cmap here.
values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
layer_values = [values[tiling.get_reflection_level(i) % len(values)] for i in range(len(tiling))]

# Convert float values to Hex strings using the YlOrBr colormap
cmap = plt.get_cmap("YlOrBr")
tiling_colors = [mcolors.to_hex(cmap(val)) for val in layer_values]

# --- New SVG Canvas Workflow --- (introduced in v1.5)
# 1. Initialize the canvas container
canvas = SVGCanvas()

# 2. Add the boundary (previously unitcircle=True)
canvas.add(UnitCircle())

# 3. Add the tiling with the pre-calculated colors
canvas.add(Tiling(tiling, facecolors=tiling_colors, edgecolor="black", lw=0.5))

# 4. Render and display 
draw_svg(canvas.render())

### Voter model

In this simple stochastic process, cells will always take on the color of the majority of their adjacent cells

In [ ]:
from hypertiling.graphics.plot import plot_tiling

import random
import matplotlib.pyplot as plt
import matplotlib.cm as cmap

#### Tiling vs Graph

For applications like finding the lowest energy state by quenching its not necessary to have coordinates. 
In this case, a graph can be used as its construction is faster for large tilings. 

However, quenching times are often magnitudes larger than the generation time even in full tiling mode. Nonetheless the advantage of reduced memory requirements holds and can be curical especially for large tilings. In fact, the memory requirement for the coordinates often is a magnitude larger than for the neighbor relations and construction. 

Prepare function which computes the energy of a configuration

In [ ]:
def get_energy(t):
    sum_ = 0
    for index in range(len(t)):
        sum_ += sum([0 if states[nbr] == states[index] else 1 for nbr in t.get_nbrs(index)])
    return sum_

Set tiling/graph parameters

In [ ]:
p, q, n = 8, 3, 10

#### Tiling

In [ ]:
t1 = time.time()
t = HyperbolicTiling(p, q, n, kernel="GRC", nbrs=True)  # Identical to HyperbolicGraph(p, q, n, kernel="GRC", tiling=True)
print(f"Generation of {len(t)} polygons took {round(time.time() - t1,5)} s")

Extract the neighbours

In [ ]:
t1 = time.time()
nbrs = t.get_nbrs_list()
print(f"Get nbrs took {round(time.time() - t1,5)} s")

Initialize the voter model with random state space

In [ ]:
states = np.random.randint(0, 2, size=len(t))
plot_tiling(t, states, cmap=cmap.Greys, edgecolor="k", cutoff=0.01, lw=0.7, clim=[0,2]);
print(f"Energy of the state is {get_energy(t):.2e}")

Run the model and display resulting configuration

In [ ]:
its = 1e5 # number of iterations

for i in range(int(its)):
    index = int(len(t) * np.random.random())
    sum_ = sum([states[nbr] for nbr in t.get_nbrs(index)])
    if sum_ >= p // 2:
        states[index] = 1
    else:
        states[index] = 0
    
plot_tiling(t, states, cmap=cmap.Greys, edgecolor="k", cutoff=0.01, lw=0.7, clim=[0,2]);
print(f"Energy of the state is {get_energy(t): .2e}")

#### Graph
When using the graph we are not able to plot the graph as we lack coordinates. However, we can use the spared memory (as we do not calculate the coordinates) to construct the graph in full mode (all sectors). This allows for faster access of neighbors.

In [ ]:
t1 = time.time()
t = HyperbolicGraph(p, q, n, kernel="GRC", sector=False)  # no coordinates
print(f"Generation of {len(t)} polygons took {round(time.time() - t1,5)} s")

In [ ]:
t1 = time.time()
nbrs = t.get_nbrs_list()
print(f"Get nbrs took {round(time.time() - t1,5)} s")

In [ ]:
states = np.random.randint(0, 2, size=len(t))
print(f"Energy of the state is {get_energy(t): .2e}")

its = 1e5 # number of iterations

for i in range(int(its)):
    index = int(len(t) * np.random.random())
    sum_ = sum([states[nbr] for nbr in t.get_nbrs(index)])
    if sum_ >= p // 2:
        states[index] = 1
    else:
        states[index] = 0
        
print(f"Energy of the state is {get_energy(t): .2e}")

## Further methods

### get_reflection_level

The natural defintion of layers of the GR kernel family is different compared to SR and Dunham kernels. Only for tilings/graphs with $q=3$ the definition coincides. To access the natural layer definition of these kernels, we provide the `get_reflection_level`method:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from hypertiling import HyperbolicTiling, TilingKernels
from hypertiling.graphics.svg import SVGCanvas, Tiling, UnitCircle, draw_svg

# some colors for the different layers
colors = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# For q != 3, the definitions do not match
t = HyperbolicTiling(3, 7, 5, kernel=TilingKernels.GRC)

# 1. Calculate the values per cell (same as before)
tiling_values = [colors[t.get_reflection_level(i) % len(colors)] for i in range(len(t))]

# 2. Convert values to Hex colors explicitly
cmap = plt.get_cmap("YlOrBr")
hex_colors = [mcolors.to_hex(cmap(val)) for val in tiling_values]

# 3. Create SVG Canvas and compose elements
canvas = SVGCanvas()
canvas.add(UnitCircle())
canvas.add(Tiling(t, facecolors=hex_colors, edgecolor="black", lw=0.5))

# 4. Render
draw_svg(canvas.render()) #

### check_integrity
For GRC, a `check_integrity` method is available if nbrs=True. As, even for tilings, the coordinates and cells are generated by a combinatoric algorithm, this method verifies whether the graph structure created by the same process is consistent. This is done in a two step process:
1. First, the number of nbrs for each cell except the last layer is verified to match p
2. Second, a bidirectional search is applied around each pair of nbrs, excluding the direct connection. This naturally should result in the paths around the two common vertices. If no path is found in the expected number of steps, the graph is considered corrupted. However, this method is only available up to the (n - (q - 1) // 2)th layer.

In [ ]:
import hypertiling.ion as ion

# adjust verbosity level to capture function output
ion.set_verbosity_level("Status")

p, q, n = 4, 5, 12

graph = HyperbolicGraph(p, q, n, kernel="GRC")
graph.check_integrity()